In [1]:
from datasets import load_dataset

ds = load_dataset("deboradum/GeoGuessr-countries")

/opt/anaconda3/envs/appliedml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
import pandas as pd
from pathlib import Path
from PIL import Image, ImageEnhance
import random
from tqdm import tqdm

In [41]:
from pathlib import Path
from PIL import Image, ImageEnhance
import random
from tqdm import tqdm


def color_jitter_pil(img, brightness=0.15, contrast=0.15, saturation=0.15):
    img = ImageEnhance.Brightness(img).enhance(random.uniform(1 - brightness, 1 + brightness))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(1 - contrast, 1 + contrast))
    img = ImageEnhance.Color(img).enhance(random.uniform(1 - saturation, 1 + saturation))
    return img


def make_offline_crops_from_dataset(
    ds,
    train_labels,
    countries,
    output_dir,
    crop_size=448,
    output_size=224,
    jpeg_quality=90,
    x_jitter=80,
    y_jitter=80,
    seed=42,
):
    random.seed(seed)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    anchors = {
        "l": 0.25,
        "c": 0.50,
        "r": 0.75,
    }

    for i, (img, label) in enumerate(
        tqdm(zip(ds["train"]["image"], train_labels), total=len(train_labels))
    ):
        country = countries[int(label)]

        country_dir = output_dir / country
        country_dir.mkdir(parents=True, exist_ok=True)

        img = img.convert("RGB")
        w, h = img.size

        if w < crop_size or h < crop_size:
            continue

        base_y = int(h / 2 - crop_size / 2)

        for crop_name, frac_x in anchors.items():
            base_x = int(frac_x * w - crop_size / 2)

            x = base_x + random.randint(-x_jitter, x_jitter)
            y = base_y + random.randint(-y_jitter, y_jitter)

            x = max(0, min(x, w - crop_size))
            y = max(0, min(y, h - crop_size))

            crop = img.crop((x, y, x + crop_size, y + crop_size))
            crop = crop.resize((output_size, output_size), Image.Resampling.LANCZOS)
            crop = color_jitter_pil(crop)

            out_path = country_dir / f"{i:08d}_{crop_name}.jpg"
            crop.save(out_path, format="JPEG", quality=jpeg_quality)

In [ ]:
def crop_bottom_center_from_directory(
    in_dir,
    output_dir,
    crop_width=200,
    crop_height=80,
    jpeg_quality=90,
    seed=42,
):
    """
    Set a 200x80 pixel region at the bottom center of each image to black and save to directory.
    Reads from input directory with country subdirectories and maintains the same structure.
    
    Args:
        in_dir: Input directory path with country subdirectories
        output_dir: Output directory path
        crop_width: Width of black region (default 200)
        crop_height: Height of black region (default 80)
        jpeg_quality: JPEG quality (default 90)
        seed: Random seed
    """
    random.seed(seed)

    in_dir = Path(in_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Iterate through country directories
    for country_dir in tqdm(in_dir.iterdir()):
        if not country_dir.is_dir():
            continue
        
        country_name = country_dir.name
        out_country_dir = output_dir / country_name
        out_country_dir.mkdir(parents=True, exist_ok=True)

        # Process all images in the country directory
        for img_path in country_dir.glob("*.jpg"):
            img = Image.open(img_path).convert("RGB")
            w, h = img.size

            if w < crop_width or h < crop_height:
                continue

            # Set bottom center region to black
            center_x = w // 2
            x = center_x - crop_width // 2
            y = h - crop_height

            # Clamp to image boundaries
            x = max(0, min(x, w - crop_width))
            y = max(0, min(y, h - crop_height))

            # Create a copy and set the region to black
            img_array = img.copy()
            black_region = Image.new("RGB", (crop_width, crop_height), (0, 0, 0))
            img_array.paste(black_region, (x, y))

            out_path = out_country_dir / img_path.name
            img_array.save(out_path, format="JPEG", quality=jpeg_quality)


In [4]:
country_to_idx = pd.read_csv("country_to_idx.csv", header=None)
country_to_idx
countries = country_to_idx[0].tolist()
print(type(countries))

<class 'list'>


In [43]:
out_dir = "/Users/august/Desktop/AppML_augmented_data/train_data"
train_labels = ds["train"]
train_labels = train_labels["label"]

make_offline_crops_from_dataset(
    ds=ds,
    train_labels=train_labels,
    countries=countries,
    output_dir=out_dir)

100%|██████████| 20394/20394 [13:35<00:00, 25.01it/s]


In [44]:
from pathlib import Path
from PIL import Image, ImageEnhance
import random
from tqdm import tqdm

def make_offline_crops_from_dataset_test(
    ds,
    train_labels,
    countries,
    output_dir,
    crop_size=448,
    output_size=224,
    jpeg_quality=90,
    x_jitter=80,
    y_jitter=80,
    seed=42,
):
    random.seed(seed)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    anchors = {
        "l": 0.25,
        "c": 0.50,
        "r": 0.75,
    }

    for i, (img, label) in enumerate(
        tqdm(zip(ds["test"]["image"], train_labels), total=len(train_labels))
    ):
        country = countries[int(label)]

        country_dir = output_dir / country
        country_dir.mkdir(parents=True, exist_ok=True)

        img = img.convert("RGB")
        w, h = img.size

        if w < crop_size or h < crop_size:
            continue

        base_y = int(h / 2 - crop_size / 2)

        for crop_name, frac_x in anchors.items():
            base_x = int(frac_x * w - crop_size / 2)

            x = base_x + random.randint(-x_jitter, x_jitter)
            y = base_y + random.randint(-y_jitter, y_jitter)

            x = max(0, min(x, w - crop_size))
            y = max(0, min(y, h - crop_size))

            crop = img.crop((x, y, x + crop_size, y + crop_size))
            crop = crop.resize((output_size, output_size), Image.Resampling.LANCZOS)
            #crop = color_jitter_pil(crop)

            out_path = country_dir / f"{i:08d}_{crop_name}.jpg"
            crop.save(out_path, format="JPEG", quality=jpeg_quality)

In [47]:
out_dir = "/Users/august/Desktop/AppML_augmented_data/test_data"
test_labels = ds["test"]
test_labels = test_labels["label"]

make_offline_crops_from_dataset_test(
    ds=ds,
    train_labels=test_labels,
    countries=countries,
    output_dir=out_dir)

100%|██████████| 5098/5098 [03:10<00:00, 26.79it/s]


In [12]:
out_dir = "/Users/august/Desktop/AppML_cut_data/train_data"
train_labels = ds["train"]
train_labels = train_labels["label"]
crop_bottom_center_from_dataset(
    ds=ds,
    labels=train_labels,
    countries=countries,
    output_dir=out_dir,
    split="train"
)

  1%|▏         | 290/20394 [00:11<12:53, 26.00it/s]


KeyboardInterrupt: 